# 🔬 소형주 팩터 롱숏 (Small-Cap Factor — 엣지가 있다면 여기)

대형주에선 팩터 알파가 통계적으로 무의미했다(롱숏 알파 t=0.72). 팩터 프리미엄은 **덜 붐비는 소형주**에서
역사적으로 더 강하다(기관 커버리지·유동성이 낮아 비효율이 남음). 여기서 마지막으로 진지하게 테스트한다.

## 🚨 반드시 이해할 한계 — 생존 편향(Survivorship Bias)
소형주는 **상장폐지·파산·인수가 매우 잦다.** 아래 유니버스는 **지금까지 살아남은** 종목들이라,
과거 백테스트는 **구조적으로 낙관적**이다(망한 종목이 빠져 있음).
→ **백테스트가 좋게 나와도 그대로 믿으면 안 된다.** 진짜 검증엔 point-in-time 구성종목 이력이 필요.

## 그 외 현실 마찰
- **거래비용↑**: 소형주는 스프레드·슬리피지가 커서 편도 0.15%로 상향
- **데이터 품질↓**: 결측·거래정지 잦음 → 일부 종목 자동 제외
- **숏 마찰↑**: 소형주 공매도는 차입 어렵고 비쌈 (여기선 분석용)

> 판정 기준은 동일: **롱숏 알파 > 0 & t-stat > 2 여야 진짜.** 아니면 소형주에서도 엣지 없음.
> ⚠️ 투자 자문 아님.

In [ ]:
# =====================================================================
# ⚙️ Cell 1: 소형/중형주 유니버스 + 데이터 (생존편향 있음에 유의)
# =====================================================================
import numpy as np, pandas as pd, yfinance as yf
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
try: plt.rcParams['font.family']='Malgun Gothic'
except Exception: pass
plt.rcParams['axes.unicode_minus']=False

# 섹터 분산 소형/중형주 (일부는 다운로드 실패/제외될 수 있음 — 정상)
UNIVERSE=['CROX','DECK','WING','SHAK','TXRH','CAKE','PLAY','BOOT','FIVE','ODP',
 'HALO','MEDP','OMCL','HAE','NVCR','TNDM','CORT','LNTH','IRTC',
 'AAON','WMS','MLI','ATKR','GTLS','EXPO','CSWI','KAI','SXI',
 'LKFN','WSFS','CATY','HOPE','FIBK','PRK','TCBK','WABC',
 'PBF','CVI','SM','MTDR','CRC','AROC',
 'MHO','LGIH','CVCO','GRBK',
 'LSCC','RMBS','POWI','SLAB','DIOD','CRUS','AMBA','SMTC','FORM','ONTO','ACLS','UCTT','COHU']
BENCH='IWM'   # 러셀2000 소형주 지수 ETF
START='2013-01-01'; END=None; COST=0.0015; ANN=252; REBAL=21; Q=0.20

def fetch_close(tickers):
    fr={}
    for t in tickers:
        try:
            df=yf.download(t,start=START,end=END,auto_adjust=True,progress=False)
            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(0)
            s=df['Close'].dropna()
            if len(s)>300: fr[t]=s
        except Exception: pass
    return pd.DataFrame(fr)

print('수집 중... (소형주라 일부 실패 가능)')
allpx=fetch_close(UNIVERSE+[BENCH])
bench=allpx[BENCH] if BENCH in allpx else allpx.iloc[:,0]
close=allpx[[c for c in allpx.columns if c!=BENCH]].dropna(axis=1,thresh=int(len(allpx)*0.8)).dropna()
bench=bench.reindex(close.index).ffill()
print(f'유효 {close.shape[1]}종목 × {close.shape[0]}일 (편도비용 {COST*100:.2f}%)')
print('⚠️ 생존편향: 이 종목들은 살아남은 것들 — 백테스트는 낙관적으로 왜곡됨')

In [ ]:
# =====================================================================
# 🧮 Cell 2: 가격 팩터 + 복합
# =====================================================================
rets=close.pct_change()
FACTORS={'Momentum':close.shift(21)/close.shift(252)-1,
         'LowVol':-(rets.rolling(126).std()),
         'Reversal':-(close.pct_change(21))}
def zrow(r):
    m,s=r.mean(),r.std(); return (r-m)/s if (s and s>0) else r*0.0
def composite(d): return sum(zrow(f.loc[d]) for f in FACTORS.values())/len(FACTORS)
print('팩터 준비 완료')

In [ ]:
# =====================================================================
# ⚖️ Cell 3: 롱숏 백테스트 + 순수 알파 회귀 (대형주와 동일 엔진)
# =====================================================================
dates=close.index; ridx=list(range(252,len(dates)-1,REBAL))
def perf(pr):
    r=np.asarray(pr,float); out={'CAGR':np.nan,'Sharpe':0.0,'MDD':0.0}
    if len(r)==0: return out
    ppy=ANN/REBAL; eq=np.cumprod(1+r); pk=np.maximum.accumulate(eq)
    out['CAGR']=float(eq[-1]**(ppy/len(r))-1) if eq[-1]>0 else np.nan
    out['Sharpe']=float(r.mean()/r.std()*np.sqrt(ppy)) if r.std()>0 else 0.0
    out['MDD']=float(((eq-pk)/pk).min()); return out

ls,lo,mk,dd=[],[],[],[]; pl,ps=set(),set()
for i in ridx:
    d=dates[i]; sc=composite(d).dropna()
    if len(sc)<10: continue
    k=max(int(len(sc)*Q),1); srt=sc.sort_values(ascending=False)
    L=list(srt.head(k).index); S=list(srt.tail(k).index); j=min(i+REBAL,len(dates)-1); d2=dates[j]
    lr=float((close.loc[d2,L]/close.loc[d,L]-1).mean()); sr=float((close.loc[d2,S]/close.loc[d,S]-1).mean())
    nl,ns=set(L),set(S); tc=((len(nl-pl)+len(pl-nl))+(len(ns-ps)+len(ps-ns)))/max(k,1)*COST
    ls.append((lr-sr)-tc); lo.append(lr-((len(nl-pl)+len(pl-nl))/max(k,1))*COST)
    mk.append(float(bench.loc[d2]/bench.loc[d]-1)); dd.append(d2); pl,ps=nl,ns
ls=pd.Series(ls,index=dd); lo=pd.Series(lo,index=dd); mk=pd.Series(mk,index=dd)

# 순수 알파 회귀
x=mk.values; y=ls.values
beta=float(np.cov(y,x)[0,1]/np.var(x)) if np.var(x)>0 else 0.0
alpha_per=float(y.mean()-beta*x.mean()); ppy=ANN/REBAL; alpha_ann=alpha_per*ppy
resid=y-(alpha_per+beta*x); se=resid.std(ddof=2)/np.sqrt(len(y)) if len(y)>2 else np.nan
t_alpha=alpha_per/se if (se and se>0) else 0.0
mh=perf((y-beta*x))

mL,mLO,mM=perf(ls.values),perf(lo.values),perf(mk.values)
print('=== 소형주 팩터 성과 (비용반영) ===')
print(f'{"":16}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}')
print(f'{"롱숏(중립)":16}{mL["CAGR"]*100:8.1f}%{mL["Sharpe"]:9.2f}{mL["MDD"]*100:8.1f}%')
print(f'{"롱온리 상위20%":16}{mLO["CAGR"]*100:8.1f}%{mLO["Sharpe"]:9.2f}{mLO["MDD"]*100:8.1f}%')
print(f'{"소형주지수(IWM)":16}{mM["CAGR"]*100:8.1f}%{mM["Sharpe"]:9.2f}{mM["MDD"]*100:8.1f}%')
print(f'\n📐 롱숏 순수 알파: {alpha_ann*100:+.2f}%/년 | t-stat {t_alpha:+.2f} | 베타 {beta:+.3f}')
if abs(t_alpha)>2 and alpha_ann>0:
    print('   → ✅ 유의한 알파! 소형주엔 팩터 엣지가 있다 (단, 생존편향 감안하면 과대평가).')
elif alpha_ann<=0:
    print('   → ⚠️ 알파 0 이하. 소형주에서도(생존편향 유리에도) 엣지 없음.')
else:
    print('   → 🤔 알파 +지만 t 약함 = 노이즈. 생존편향까지 감안하면 실질 엣지 의심.')

In [ ]:
# =====================================================================
# 📈 Cell 4: 자산곡선 & 낙폭
# =====================================================================
fig,ax=plt.subplots(2,1,figsize=(13,9))
(1+ls).cumprod().plot(ax=ax[0],label='롱숏(중립)',lw=2.4,color='purple')
(1+lo).cumprod().plot(ax=ax[0],label='롱온리 상위20%',lw=1.4,color='navy')
(1+mk).cumprod().plot(ax=ax[0],label='소형주지수(IWM)',lw=1.4,ls='--',color='gray')
ax[0].set_yscale('log'); ax[0].set_title('소형주 팩터 — 자산곡선(로그) [생존편향 유의]')
ax[0].legend(); ax[0].grid(alpha=0.3)
for nm,sr in [('롱숏',ls),('롱온리',lo),('지수',mk)]:
    eq=(1+sr).cumprod(); (eq/eq.cummax()-1).mul(100).plot(ax=ax[1],label=nm,lw=1.4)
ax[1].set_title('낙폭(Drawdown, %)'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 🧾 해석 & 프로젝트 결말

### 판정 (생존편향 보정해서 엄격하게)
- **알파 t>2 & 양수** 라도, **생존편향이 이미 유리하게 부풀린** 값이다. 실제는 이보다 나쁘다.
  즉 t가 겨우 2를 넘는 정도면 실질 엣지는 의심스럽다. **t가 확실히 크고(>3) 알파가 커야** 겨우 신뢰.
- **알파 ≈ 0 또는 t 약함** → 소형주에서조차(편향 유리에도) 엣지 없음 → **개인은 이 데이터로 엣지 못 잡는다**가 최종 결론.

### 만약 여기서도 엣지가 안 나오면 (그럴 가능성 큼)
그건 **완결된 정직한 결론**이다:
> "무료 데이터 + 공개 가격/펀더멘털로는, 개인이 통계적으로 유의한 예측 엣지를 찾기 어렵다.
>  진짜 엣지는 (1) 독점적 데이터, (2) 초단타 인프라, (3) 덜 접근 가능한 시장(사모·크립토 등),
>  또는 (4) 규율 있는 리스크 관리(추세추종) 에 있다."

### 그래도 더 간다면 (엣지 탐구의 마지막 프론티어)
1. **PIT 구성종목 이력** 확보로 생존편향 제거 (유료 데이터·상당한 작업)
2. **크립토** — 24/7, 비효율 많음, 리테일 접근 가능 (고위험)
3. **이벤트 드리븐**(어닝 서프라이즈 PEAD) — 느리게 반영되는 정보

> 이 프로젝트의 진짜 성과: **엄격한 검증으로 "쉬운 엣지는 없다"를 스스로 증명하고,
> 그 위에 정직한 리스크관리 시스템(추세추종 + 페이퍼 자동매매)을 세운 것.**